# Pipeline

> Run OCR + fix the markdown headings + describe images/figures

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
from fastcore.all import *
from mistocr.core import read_pgs, ocr_pdf
from mistocr.refine import add_img_descs, fix_hdgs
from pathlib import Path
from asyncio import Semaphore, gather, sleep
import os, json, shutil

In [ ]:
#| export
@delegates(add_img_descs)
async def pdf_to_md(
    pdf_path:str, # Path to input PDF file
    dst:str, # Destination directory for output markdown
    ocr_output:str=None, # Optional OCR output directory (defaults to pdf_path stem)
    model:str='claude-sonnet-4-5', # Model to use for heading fixes and image descriptions
    add_img_desc:bool=True, # Whether to add image descriptions
    progress:bool=True, # Whether to show progress messages
    **kwargs):
    "Convert PDF to markdown with OCR, fixed heading hierarchy, and optional image descriptions"
    ocr_dir = Path(ocr_output) if ocr_output else Path(pdf_path).with_suffix('')
    n_steps = 3 if add_img_desc else 2
    if progress: print(f"Step 1/{n_steps}: Running OCR on {pdf_path}...")
    ocr_pdf(pdf_path, ocr_dir)
    if progress: print(f"Step 2/{n_steps}: Fixing heading hierarchy...")
    fix_hdgs(ocr_dir, model=model)
    if add_img_desc:
        if progress: print(f"Step 3/{n_steps}: Adding image descriptions...")
        await add_img_descs(ocr_dir, dst=dst, model=model, progress=progress, **kwargs)
    elif dst and Path(dst) != ocr_dir:
        shutil.copytree(ocr_dir, dst, dirs_exist_ok=True)
    if progress: print("Done!")